In [1]:
from pathlib import Path
import fitz  # PyMuPDF
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()

RAW_PDF_DIR = PROJECT_ROOT / "data" / "raw_pdfs"
LABEL_DIR = PROJECT_ROOT / "data" / "labels"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = LABEL_DIR / "manual_labels_v1.csv"

In [2]:
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())

df.head()

Shape: (175, 19)
['document_name', 'page_number', 'char_count', 'word_count', 'avg_word_length', 'printable_char_ratio', 'whitespace_ratio', 'alphabetic_ratio', 'digit_ratio', 'symbol_ratio', 'text_block_count', 'image_count', 'page_width', 'page_height', 'page_area', 'chars_per_page_area', 'words_per_page_area', 'text_preview', 'label']


,document_name,page_number,char_count,word_count,avg_word_length,printable_char_ratio,whitespace_ratio,alphabetic_ratio,digit_ratio,symbol_ratio,text_block_count,image_count,page_width,page_height,page_area,chars_per_page_area,words_per_page_area,text_preview,label
0,anime_ir_report_80.pdf,1,2182,304,5.953947,0.999542,0.129239,0.807058,0.021082,0.042621,37,0,594.959961,841.919983,500908.6801,0.004356,0.000607,Anime Information Retrieval System CS 429 – In...,0
1,anime_ir_report_80.pdf,2,2032,244,6.102459,0.973917,0.183563,0.701772,0.028543,0.086122,28,0,594.959961,841.919983,500908.6801,0.004057,0.000487,Salton & McGill (1983): Vector Space Model and...,0
2,anime_ir_report_80.pdf,3,1671,227,5.744493,0.989228,0.144225,0.742071,0.027528,0.086176,34,0,594.959961,841.919983,500908.6801,0.003336,0.000453,Tokenized articles → Word2Vec training (5 epoc...,0
3,anime_ir_report_80.pdf,4,2090,270,5.570370,0.961244,0.154545,0.679426,0.025359,0.140670,18,0,594.959961,841.919983,500908.6801,0.004172,0.000539,Methods: - validate_query(query_text) → erro...,0
4,anime_ir_report_80.pdf,5,2203,318,5.449686,0.994553,0.121198,0.714934,0.055379,0.108488,37,0,594.959961,841.919983,500908.6801,0.004398,0.000635,MODE B: Inside Notebook (Minimal Parameters) C...,0


In [3]:
def safe_area(x0, y0, x1, y1):
    width = max(0, x1 - x0)
    height = max(0, y1 - y0)
    return width * height


def extract_layout_features_for_page(page):
    page_width = page.rect.width
    page_height = page.rect.height
    page_area = page_width * page_height

    text_blocks = page.get_text("blocks")
    image_info = page.get_images(full=True)

    text_areas = []

    for block in text_blocks:
        x0, y0, x1, y1 = block[:4]
        area = safe_area(x0, y0, x1, y1)
        text_areas.append(area)

    total_text_area = sum(text_areas)
    largest_text_block_area = max(text_areas) if text_areas else 0
    avg_text_block_area = np.mean(text_areas) if text_areas else 0

    image_areas = []

    image_blocks = [
        b for b in page.get_text("dict").get("blocks", [])
        if b.get("type") == 1
    ]

    for block in image_blocks:
        bbox = block.get("bbox", None)

        if bbox:
            x0, y0, x1, y1 = bbox
            image_areas.append(safe_area(x0, y0, x1, y1))

    total_image_area = sum(image_areas)
    largest_image_area = max(image_areas) if image_areas else 0
    avg_image_area = np.mean(image_areas) if image_areas else 0

    text_area_ratio = total_text_area / page_area if page_area else 0
    image_area_ratio = total_image_area / page_area if page_area else 0
    largest_image_area_ratio = largest_image_area / page_area if page_area else 0
    largest_text_block_ratio = largest_text_block_area / page_area if page_area else 0

    text_to_image_area_ratio = (
        text_area_ratio / image_area_ratio
        if image_area_ratio > 0
        else text_area_ratio
    )

    return {
        "layout_page_width": page_width,
        "layout_page_height": page_height,
        "layout_page_area": page_area,

        "layout_text_block_count": len(text_blocks),
        "layout_image_count": len(image_info),

        "total_text_area": total_text_area,
        "avg_text_block_area": avg_text_block_area,
        "largest_text_block_area": largest_text_block_area,

        "total_image_area": total_image_area,
        "avg_image_area": avg_image_area,
        "largest_image_area": largest_image_area,

        "text_area_ratio": text_area_ratio,
        "image_area_ratio": image_area_ratio,
        "largest_image_area_ratio": largest_image_area_ratio,
        "largest_text_block_ratio": largest_text_block_ratio,
        "text_to_image_area_ratio": text_to_image_area_ratio,
    }

In [4]:
layout_rows = []

for _, row in df.iterrows():
    pdf_path = RAW_PDF_DIR / row["document_name"]
    page_number = int(row["page_number"])

    if not pdf_path.exists():
        print("Missing PDF:", pdf_path)
        continue

    doc = fitz.open(pdf_path)

    page_index = page_number - 1

    if page_index < 0 or page_index >= len(doc):
        print("Invalid page:", row["document_name"], page_number)
        doc.close()
        continue

    page = doc[page_index]

    features = extract_layout_features_for_page(page)

    features["document_name"] = row["document_name"]
    features["page_number"] = page_number

    layout_rows.append(features)

    doc.close()

layout_df = pd.DataFrame(layout_rows)

print("Layout feature shape:", layout_df.shape)
layout_df.head()

Layout feature shape: (175, 18)


,layout_page_width,layout_page_height,layout_page_area,layout_text_block_count,layout_image_count,total_text_area,avg_text_block_area,largest_text_block_area,total_image_area,avg_image_area,largest_image_area,text_area_ratio,image_area_ratio,largest_image_area_ratio,largest_text_block_ratio,text_to_image_area_ratio,document_name,page_number
0,594.959961,841.919983,500908.680145,37,0,119147.171549,3220.193826,6583.045756,0.0,0.0,0.0,0.237862,0.0,0.0,0.013142,0.237862,anime_ir_report_80.pdf,1
1,594.959961,841.919983,500908.680145,28,0,129813.527751,4636.197420,61616.470045,0.0,0.0,0.0,0.259156,0.0,0.0,0.123009,0.259156,anime_ir_report_80.pdf,2
2,594.959961,841.919983,500908.680145,34,0,94381.440326,2775.924715,15624.390303,0.0,0.0,0.0,0.188420,0.0,0.0,0.031192,0.188420,anime_ir_report_80.pdf,3
3,594.959961,841.919983,500908.680145,18,0,156292.198482,8682.899916,61043.886425,0.0,0.0,0.0,0.312017,0.0,0.0,0.121866,0.312017,anime_ir_report_80.pdf,4
4,594.959961,841.919983,500908.680145,37,0,118970.502892,3215.418997,22626.932874,0.0,0.0,0.0,0.237509,0.0,0.0,0.045172,0.237509,anime_ir_report_80.pdf,5


In [5]:
enhanced_df = df.merge(
    layout_df,
    on=["document_name", "page_number"],
    how="left"
)

print("Enhanced shape:", enhanced_df.shape)
enhanced_df.head()

Enhanced shape: (175, 35)


,document_name,page_number,char_count,word_count,avg_word_length,printable_char_ratio,whitespace_ratio,alphabetic_ratio,digit_ratio,symbol_ratio,...,avg_text_block_area,largest_text_block_area,total_image_area,avg_image_area,largest_image_area,text_area_ratio,image_area_ratio,largest_image_area_ratio,largest_text_block_ratio,text_to_image_area_ratio
0,anime_ir_report_80.pdf,1,2182,304,5.953947,0.999542,0.129239,0.807058,0.021082,0.042621,...,3220.193826,6583.045756,0.0,0.0,0.0,0.237862,0.0,0.0,0.013142,0.237862
1,anime_ir_report_80.pdf,2,2032,244,6.102459,0.973917,0.183563,0.701772,0.028543,0.086122,...,4636.197420,61616.470045,0.0,0.0,0.0,0.259156,0.0,0.0,0.123009,0.259156
2,anime_ir_report_80.pdf,3,1671,227,5.744493,0.989228,0.144225,0.742071,0.027528,0.086176,...,2775.924715,15624.390303,0.0,0.0,0.0,0.188420,0.0,0.0,0.031192,0.188420
3,anime_ir_report_80.pdf,4,2090,270,5.570370,0.961244,0.154545,0.679426,0.025359,0.140670,...,8682.899916,61043.886425,0.0,0.0,0.0,0.312017,0.0,0.0,0.121866,0.312017
4,anime_ir_report_80.pdf,5,2203,318,5.449686,0.994553,0.121198,0.714934,0.055379,0.108488,...,3215.418997,22626.932874,0.0,0.0,0.0,0.237509,0.0,0.0,0.045172,0.237509


In [6]:
output_path = PROCESSED_DIR / "manual_labels_with_layout_features_v1.csv"

enhanced_df.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: C:\Users\rudra\Documents\VS Code Files\DocuMindAI\data\processed\manual_labels_with_layout_features_v1.csv
